# Vectors


## Setup


In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / ".git").exists()), cwd)
load_dotenv(repo_root / ".env", override=True)
sys.path.insert(0, str(repo_root / "W5-RAG"))

import numpy as np
import tiktoken
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

from store import EMBEDDING_MODEL, MINILM, clear, client, knowledge_base

print(EMBEDDING_MODEL, "is the Azure embedding used on Day 3")
print("this notebook embeds with all-MiniLM-L6-v2")


text-embedding-3-large is the Azure embedding used on Day 3
this notebook embeds with all-MiniLM-L6-v2


## Size


In [2]:
files = sorted(knowledge_base.rglob("*.md"))
text = "\n\n".join(path.read_text(encoding="utf-8") for path in files)
encoding = tiktoken.encoding_for_model("gpt-4o")
print(f"{len(files)} files")
print(f"{len(text):,} characters")
print(f"{len(encoding.encode(text)):,} tokens")


76 files
304,432 characters
63,555 tokens


## Load


In [3]:
documents = []
for folder in sorted(path for path in knowledge_base.iterdir() if path.is_dir()):
    for file in sorted(folder.glob("*.md")):
        documents.append(Document(
            page_content=file.read_text(encoding="utf-8"),
            metadata={"source": str(file), "doc_type": folder.name},
        ))

print(f"Loaded {len(documents)} documents")
print(documents[0].metadata)


Loaded 76 documents
{'source': '/home/akalanka/SSD/Akalanka/cources/udemy-agentic-ai-core-track/W5-RAG/knowledge-base/company/about.md', 'doc_type': 'company'}


## Chunks

1000 characters, overlap 200. The Azure store on Day 3 uses 500 / 200.


In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(documents)
print(f"Divided into {len(chunks)} chunks")
print(chunks[0].page_content[:300])


Divided into 413 chunks
# About Insurellm

Insurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.

The company experienced rapid growth in its


## Chroma


In [5]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
clear(MINILM)

from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    client=client(),
    collection_name=MINILM,
)
collection = client().get_collection(MINILM)
sample = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
print(f"{collection.count()} vectors, {len(sample)} dimensions")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

413 vectors, 384 dimensions


## Plot


In [6]:
palette = {
    "products": "blue",
    "employees": "green",
    "contracts": "red",
    "company": "orange",
}
result = collection.get(include=["embeddings", "documents", "metadatas"])
vectors = np.array(result["embeddings"])
doc_types = [item["doc_type"] for item in result["metadatas"]]
colors = [palette[item] for item in doc_types]
hover = [
    f"Type: {kind}<br>Text: {doc[:100]}..."
    for kind, doc in zip(doc_types, result["documents"])
]
print(vectors.shape)

perplexity = min(30, max(2, len(vectors) - 1))
reduced = TSNE(n_components=2, random_state=42, perplexity=perplexity).fit_transform(vectors)
figure = go.Figure(data=[go.Scatter(
    x=reduced[:, 0],
    y=reduced[:, 1],
    mode="markers",
    marker=dict(size=5, color=colors, opacity=0.8),
    text=hover,
    hoverinfo="text",
)])
figure.update_layout(title="2D Chroma vectors", width=800, height=600)
figure.show()

reduced = TSNE(n_components=3, random_state=42, perplexity=perplexity).fit_transform(vectors)
figure = go.Figure(data=[go.Scatter3d(
    x=reduced[:, 0],
    y=reduced[:, 1],
    z=reduced[:, 2],
    mode="markers",
    marker=dict(size=4, color=colors, opacity=0.8),
    text=hover,
    hoverinfo="text",
)])
figure.update_layout(title="3D Chroma vectors", width=800, height=600)
figure.show()


(413, 384)
